In [1]:
# Install correct version (if needed)
!pip install -q kaggle-environments>=0.1.6

In [2]:
import numpy as np
import random
import math
import inspect
from kaggle_environments import make, evaluate

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 16.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_go
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_goofspiel
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_hearts
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_matching_pennies_3p
[kaggle_environments.envs.

In [3]:
# Create environment
env = make("connectx", debug=True)
env.render(mode="ansi")

'+---+---+---+---+---+---+---+\n| 0 | 0 | 0 | 0 | 0 | 0 | 0 |\n+---+---+---+---+---+---+---+\n| 0 | 0 | 0 | 0 | 0 | 0 | 0 |\n+---+---+---+---+---+---+---+\n| 0 | 0 | 0 | 0 | 0 | 0 | 0 |\n+---+---+---+---+---+---+---+\n| 0 | 0 | 0 | 0 | 0 | 0 | 0 |\n+---+---+---+---+---+---+---+\n| 0 | 0 | 0 | 0 | 0 | 0 | 0 |\n+---+---+---+---+---+---+---+\n| 0 | 0 | 0 | 0 | 0 | 0 | 0 |\n+---+---+---+---+---+---+---+\n'

In [4]:
# ==============================
# Strong Minimax Agent
# ==============================

def strong_agent(obs, conf):
    ROWS = conf.rows
    COLS = conf.columns

    board = [obs.board[i*COLS:(i+1)*COLS] for i in range(ROWS)]
    mark = obs.mark
    opp = 2 if mark == 1 else 1
    MAX_DEPTH = 5

    def valid_moves(board):
        return [c for c in range(COLS) if board[0][c] == 0]

    def drop(board, col, piece):
        temp = [row[:] for row in board]
        for r in range(ROWS-1, -1, -1):
            if temp[r][col] == 0:
                temp[r][col] = piece
                break
        return temp

    def winning(board, piece):
        for r in range(ROWS):
            for c in range(COLS-3):
                if all(board[r][c+i] == piece for i in range(4)):
                    return True

        for c in range(COLS):
            for r in range(ROWS-3):
                if all(board[r+i][c] == piece for i in range(4)):
                    return True

        for r in range(ROWS-3):
            for c in range(COLS-3):
                if all(board[r+i][c+i] == piece for i in range(4)):
                    return True

        for r in range(3, ROWS):
            for c in range(COLS-3):
                if all(board[r-i][c+i] == piece for i in range(4)):
                    return True

        return False

    def score_window(window, piece):
        score = 0

        if window.count(piece) == 4:
            score += 100
        elif window.count(piece) == 3 and window.count(0) == 1:
            score += 15
        elif window.count(piece) == 2 and window.count(0) == 2:
            score += 5

        if window.count(opp) == 3 and window.count(0) == 1:
            score -= 12

        return score

    def score(board, piece):
        total = 0

        center = [board[r][COLS//2] for r in range(ROWS)]
        total += center.count(piece) * 6

        for r in range(ROWS):
            for c in range(COLS - 3):
                total += score_window([board[r][c+i] for i in range(4)], piece)

        for c in range(COLS):
            for r in range(ROWS - 3):
                total += score_window([board[r+i][c] for i in range(4)], piece)

        for r in range(ROWS - 3):
            for c in range(COLS - 3):
                total += score_window([board[r+i][c+i] for i in range(4)], piece)

        for r in range(3, ROWS):
            for c in range(COLS - 3):
                total += score_window([board[r-i][c+i] for i in range(4)], piece)

        return total
    def minimax(board, depth, alpha, beta, maximizing):
        valid = valid_moves(board)
        # Hamleleri merkeze yakınlığa göre sıralayarak Alpha-Beta verimini artır
        valid.sort(key=lambda c: abs(COLS//2 - c)) 
        valid = sorted(valid_moves(board), key=lambda c: abs(COLS//2 - c))
        terminal = winning(board, mark) or winning(board, opp) or len(valid) == 0

        if depth == 0 or terminal:
            if winning(board, mark):
                return None, 1000000
            elif winning(board, opp):
                return None, -1000000
            else:
                return None, score(board, mark)

        if maximizing:
            value = -math.inf
            best_col = random.choice(valid)
            for col in valid:
                new_board = drop(board, col, mark)
                _, new_score = minimax(new_board, depth-1, alpha, beta, False)
                if new_score > value:
                    value = new_score
                    best_col = col
                alpha = max(alpha, value)
                if alpha >= beta:
                    break
            return best_col, value
        else:
            value = math.inf
            best_col = random.choice(valid)
            for col in valid:
                new_board = drop(board, col, opp)
                _, new_score = minimax(new_board, depth-1, alpha, beta, True)
                if new_score < value:
                    value = new_score
                    best_col = col
                beta = min(beta, value)
                if alpha >= beta:
                    break
            return best_col, value

    # Immediate win
    for col in valid_moves(board):
        if winning(drop(board, col, mark), mark):
            return col

    # Immediate block
    for col in valid_moves(board):
        if winning(drop(board, col, opp), opp):
            return col

    col, _ = minimax(board, MAX_DEPTH, -math.inf, math.inf, True)
    return col

In [5]:
# ==============================
# Run & Evaluate
# ==============================

env.reset()
env.run([strong_agent, "negamax"])
env.render(mode="ansi")

print("Vs Random:", evaluate("connectx", [strong_agent, "random"], num_episodes=10))
print("Vs Negamax:", evaluate("connectx", [strong_agent, "negamax"], num_episodes=10))

Vs Random: [[1, -1], [1, -1], [1, -1], [1, -1], [1, -1], [1, -1], [1, -1], [1, -1], [1, -1], [1, -1]]
Vs Negamax: [[1, -1], [1, -1], [1, -1], [1, -1], [1, -1], [1, -1], [1, -1], [1, -1], [1, -1], [1, -1]]


In [6]:
# ==============================
# Create submission file
# ==============================

with open("submission.py", "w") as f:
    f.write("import numpy as np\n")
    f.write("import random\n")
    f.write("import math\n\n")
    f.write(inspect.getsource(strong_agent))

print("submission.py created successfully!")

# Self-play test
env = make("connectx", debug=True)
env.run(["submission.py", "submission.py"])
env.render(mode="ansi")

print(env.state[0].status, env.state[1].status)

submission.py created successfully!
DONE DONE
